# Testing Complete Multipartite Schur-Coefficient Formulas

## Preliminaries

In [1]:
# Imports

In [2]:
# Define Schur Basis

s = SymmetricFunctions(QQ).s()

### Helper Functions

In [3]:
# Function to compute CSF (in Schur basis) for any complete multipartite graph

def complete_multipartite_schur(parts):
    
    '''
    Input: a partition lambda such as [2,2,2]
    Output: the chromatic symmetric function of K_{lambda} expanded in the Schur basis
    '''
    
    G = graphs.CompleteMultipartiteGraph(parts)
    X = G.chromatic_symmetric_function()
    s = SymmetricFunctions(QQ).s()
    return s(X)

In [4]:
# Function to generate poset P such that inc(P) = K_{(2^n)}

def disjoint_two_chains(n):

    '''
    Outputs a poset with n disjoint chains of length 2
    '''
    
    elements = []
    covers = []
    
    for i in range(n):
        bottom = (i, 0)
        top = (i, 1)
        elements += [bottom, top]
        covers.append((bottom, top))   # bottom < top
    
    return Poset((elements, covers), cover_relations=True)

In [9]:
# Function to generate poset P such that inc(P) = K_{(3,2^n)}

def three_and_disjoint_two_chains(n):

    '''
    Outputs a poset with one chain of length 3 and n disjoint chains of length 2
    '''
    
    elements = []
    covers = []
    
    # chain of length 3
    chain3 = [('a', 0), ('a', 1), ('a', 2)]
    elements += chain3
    covers.append((chain3[0], chain3[1]))   # ('a',0) < ('a',1)
    covers.append((chain3[1], chain3[2]))   # ('a',1) < ('a',2)
    
    # n disjoint chains of length 2
    for i in range(n):
        bottom = (i, 0)
        top = (i, 1)
        elements += [bottom, top]
        covers.append((bottom, top))   # bottom < top
    
    return Poset((elements, covers), cover_relations=True)

In [10]:
# Function to generate poset P such that inc(P) = K_{(2^n,1)}

def disjoint_two_chains_and_singleton(n):

    '''
    Outputs a poset with n disjoint chains of length 2 and one singleton
    '''
    
    elements = []
    covers = []
    
    # n disjoint chains of length 2
    for i in range(n):
        bottom = (i, 0)
        top = (i, 1)
        elements += [bottom, top]
        covers.append((bottom, top))   # bottom < top
    
    # singleton
    singleton = ('s', 0)
    elements.append(singleton)
    
    return Poset((elements, covers), cover_relations=True)

In [5]:
# Function to count non-increasing paths for a poset

def N_sp_from_poset(P):
    
    """
    Count spanning non-increasing sequences in inc(P).
    A spanning non-increasing sequence is a permutation (v1,...,vn)
    such that for each i, NOT (vi <= v_{i+1}) in the poset.
    By convention, if P has 0 elements, return 1.
    """
    
    V = list(P)
    n = len(V)
    if n == 0:
        return Integer(1)
    
    count = Integer(0)
    for perm in Permutations(V):
        ok = True
        for i in range(n - 1):
            if P.le(perm[i], perm[i+1]):  # violates non-increasing condition
                ok = False
                break
        if ok:
            count += 1
    return count

In [6]:
# Function to check lemma formula for Schur coefficients of 2^{beta}

def lemma_rhs(beta, C):
    
    """
    Compute
        beta!/(beta-C)! * N_sp(K_(2^(beta-C))).
    """
    
    m = beta - C
    return factorial(beta) // factorial(beta - C) * N_sp_from_poset(disjoint_two_chains(m))

In [7]:
# Function to verify two beta lemma for a particular graph

def verify_two_beta_lemma(beta, verbose=True):
    
    """
    Check the lemma for Schur coefficients of 2^{beta} for a fixed beta and all C = 0,...,beta.
    Returns True if all checks pass.
    """
    
    Xs = complete_multipartite_schur([2]*beta)
    coeffs_dict = Xs.monomial_coefficients()
    
    all_ok = True
    
    for C in range(beta + 1):
        D = 2*beta - 2*C
        mu = Partition([2]*C + [1]*D)
        
        lhs = coeffs_dict.get(mu, Integer(0))
        rhs = factorial(beta) // factorial(beta - C) * N_sp_from_poset(disjoint_two_chains(beta - C))
        
        if verbose:
            print(f"beta = {beta}, C = {C}, D = {D}, mu = {mu}")
            print(f"  lhs = {lhs}")
            print(f"  rhs = {rhs}")
            print(f"  match? {lhs == rhs}")
            print()
        
        if lhs != rhs:
            all_ok = False
    
    return all_ok

In [8]:
# Function to verify two beta lemma for multiple small values of beta

def verify_two_beta_lemma_up_to(max_beta, verbose=False):
    
    """
    Verify the lemma for beta = 0,1,...,max_beta.
    Returns True if all checks pass.
    """
    
    for beta in range(max_beta + 1):
        ok = verify_two_beta_lemma(beta, verbose=verbose)
        if not ok:
            print(f"FAILED at beta = {beta}")
            return False
        else:
            print(f"Passed beta = {beta}")
    return True

In [14]:
# Function to verify K_(3,2^beta) theorem for a particular graph

def verify_K32_theorem(beta, verbose=True):
    
    """
    Check the theorem for Schur coefficients of K_(3,2^beta) for a fixed beta.
    Verifies the three cases appearing in the theorem.
    
    Returns True if all checks pass.
    """
    
    Xs = complete_multipartite_schur([3] + [2]*beta)
    coeffs_dict = Xs.monomial_coefficients()
    
    all_ok = True
    
    # --------------------------------------------------
    # Case 1:
    # [s_(3,2^C,1^D)] X_{K_(3,2^beta)}
    # = beta!/(beta-C)! * N_sp(K_(2^(beta-C)))
    # where 2C + D = 2beta
    # --------------------------------------------------
    
    if verbose:
        print("Case 1:")
    
    for C in range(beta + 1):
        D = 2*beta - 2*C
        mu = Partition([3] + [2]*C + [1]*D)
        
        lhs = coeffs_dict.get(mu, Integer(0))
        rhs = factorial(beta) // factorial(beta - C) * N_sp_from_poset(disjoint_two_chains(beta - C))
        
        if verbose:
            print(f"beta = {beta}, C = {C}, D = {D}, mu = {mu}")
            print(f"  lhs = {lhs}")
            print(f"  rhs = {rhs}")
            print(f"  match? {lhs == rhs}")
            print()
        
        if lhs != rhs:
            all_ok = False
    
    # --------------------------------------------------
    # Case 2:
    # [s_(2^C,1^D)] X_{K_(3,2^beta)}
    # = beta!/(beta-C+1)! * ( (beta-C+1)N_sp(K_(3,2^(beta-C)))
    #                         - N_sp(K_(2^(beta-C+1)))
    #                         + (C+2)N_sp(K_(2^(beta-C+1),1)) )
    #
    # where C >= 1, D >= 1, and 2C + D = 2beta + 3
    # --------------------------------------------------
    
    if verbose:
        print("Case 2:")
    
    for C in range(1, beta + 2):
        D = 2*beta + 3 - 2*C
        
        if D < 1:
            continue
        
        mu = Partition([2]*C + [1]*D)
        lhs = coeffs_dict.get(mu, Integer(0))
        
        # Theorem convention: if beta < C, then N_sp(K_(3,2^(beta-C)))) = 0
        if beta < C:
            term1 = Integer(0)
        else:
            term1 = N_sp_from_poset(three_and_disjoint_two_chains(beta - C))
        
        term2 = N_sp_from_poset(disjoint_two_chains(beta - C + 1))
        term3 = N_sp_from_poset(disjoint_two_chains_and_singleton(beta - C + 1))
        
        rhs = (factorial(beta) // factorial(beta - C + 1)) * (
            (beta - C + 1) * term1
            - term2
            + (C + 2) * term3
        )
        
        if verbose:
            print(f"beta = {beta}, C = {C}, D = {D}, mu = {mu}")
            print(f"  lhs = {lhs}")
            print(f"  rhs = {rhs}")
            print(f"  match? {lhs == rhs}")
            print()
        
        if lhs != rhs:
            all_ok = False
    
    # --------------------------------------------------
    # Case 3:
    # [s_(1^(2beta+3))] X_{K_(3,2^beta)}
    # = N_sp(K_(3,2^beta))
    # --------------------------------------------------
    
    if verbose:
        print("Case 3:")
    
    mu = Partition([1]*(2*beta + 3))
    lhs = coeffs_dict.get(mu, Integer(0))
    rhs = N_sp_from_poset(three_and_disjoint_two_chains(beta))
    
    if verbose:
        print(f"beta = {beta}, mu = {mu}")
        print(f"  lhs = {lhs}")
        print(f"  rhs = {rhs}")
        print(f"  match? {lhs == rhs}")
        print()
    
    if lhs != rhs:
        all_ok = False
    
    return all_ok

In [19]:
# Function to verify K_(3,2^beta) theorem up to a given beta

def verify_K32_theorem_up_to(max_beta, verbose=False):
    
    """
    Verify the theorem for beta = 1,2,...,max_beta.
    Returns True if all checks pass.
    """
    
    for beta in range(1, max_beta + 1):
        ok = verify_K32_theorem(beta, verbose=verbose)
        
        if not ok:
            print(f"FAILED at beta = {beta}")
            return False
        else:
            print(f"Passed beta = {beta}")
            print('''

            *********************************************************************************
            
            ''')
    return True

## Tests

### $K_{(2^{\beta})}$ Tests

In [10]:
# Example CSF of complete multipartite graph

X = complete_multipartite_schur([2,2,2])
print(X)

426*s[1, 1, 1, 1, 1, 1] + 42*s[2, 1, 1, 1, 1] + 6*s[2, 2, 1, 1] + 6*s[2, 2, 2]


In [12]:
# Verify two beta lemma up to beta = 6

verify_two_beta_lemma_up_to(6, verbose = True)

beta = 0, C = 0, D = 0, mu = []
  lhs = 1
  rhs = 1
  match? True

Passed beta = 0
beta = 1, C = 0, D = 2, mu = [1, 1]
  lhs = 1
  rhs = 1
  match? True

beta = 1, C = 1, D = 0, mu = [2]
  lhs = 1
  rhs = 1
  match? True

Passed beta = 1
beta = 2, C = 0, D = 4, mu = [1, 1, 1, 1]
  lhs = 14
  rhs = 14
  match? True

beta = 2, C = 1, D = 2, mu = [2, 1, 1]
  lhs = 2
  rhs = 2
  match? True

beta = 2, C = 2, D = 0, mu = [2, 2]
  lhs = 2
  rhs = 2
  match? True

Passed beta = 2
beta = 3, C = 0, D = 6, mu = [1, 1, 1, 1, 1, 1]
  lhs = 426
  rhs = 426
  match? True

beta = 3, C = 1, D = 4, mu = [2, 1, 1, 1, 1]
  lhs = 42
  rhs = 42
  match? True

beta = 3, C = 2, D = 2, mu = [2, 2, 1, 1]
  lhs = 6
  rhs = 6
  match? True

beta = 3, C = 3, D = 0, mu = [2, 2, 2]
  lhs = 6
  rhs = 6
  match? True

Passed beta = 3
beta = 4, C = 0, D = 8, mu = [1, 1, 1, 1, 1, 1, 1, 1]
  lhs = 24024
  rhs = 24024
  match? True

beta = 4, C = 1, D = 6, mu = [2, 1, 1, 1, 1, 1, 1]
  lhs = 1704
  rhs = 1704
  match? Tru

KeyboardInterrupt: 

### $K_{(3,2^{\beta})}$ Tests

In [13]:
# Confirm poset functions are working

P1 = three_and_disjoint_two_chains(2)
print(P1)
print(P1.incomparability_graph().is_isomorphic(graphs.CompleteMultipartiteGraph([3,2,2])))

P2 = disjoint_two_chains_and_singleton(3)
print(P2)
print(P2.incomparability_graph().is_isomorphic(graphs.CompleteMultipartiteGraph([2,2,2,1])))

Finite poset containing 7 elements
True
Finite poset containing 7 elements
True


In [16]:
# Test function to verify K32 theorem

verify_K32_theorem(2, verbose=True)

Case 1:
beta = 2, C = 0, D = 4, mu = [3, 1, 1, 1, 1]
  lhs = 14
  rhs = 14
  match? True

beta = 2, C = 1, D = 2, mu = [3, 2, 1, 1]
  lhs = 2
  rhs = 2
  match? True

beta = 2, C = 2, D = 0, mu = [3, 2, 2]
  lhs = 2
  rhs = 2
  match? True

Case 2:
beta = 2, C = 1, D = 5, mu = [2, 1, 1, 1, 1, 1]
  lhs = 312
  rhs = 312
  match? True

beta = 2, C = 2, D = 3, mu = [2, 2, 1, 1, 1]
  lhs = 32
  rhs = 32
  match? True

beta = 2, C = 3, D = 1, mu = [2, 2, 2, 1]
  lhs = 8
  rhs = 8
  match? True

Case 3:
beta = 2, mu = [1, 1, 1, 1, 1, 1, 1]
  lhs = 2286
  rhs = 2286
  match? True



True

In [ ]:
# Verify K32 theorem up to beta = 5 (13 vertices)

verify_K32_theorem_up_to(5, verbose=True)

Case 1:
beta = 1, C = 0, D = 2, mu = [3, 1, 1]
  lhs = 1
  rhs = 1
  match? True

beta = 1, C = 1, D = 0, mu = [3, 2]
  lhs = 1
  rhs = 1
  match? True

Case 2:
beta = 1, C = 1, D = 3, mu = [2, 1, 1, 1]
  lhs = 12
  rhs = 12
  match? True

beta = 1, C = 2, D = 1, mu = [2, 2, 1]
  lhs = 3
  rhs = 3
  match? True

Case 3:
beta = 1, mu = [1, 1, 1, 1, 1]
  lhs = 46
  rhs = 46
  match? True

Passed beta = 1


            *********************************************************************************
            
            
Case 1:
beta = 2, C = 0, D = 4, mu = [3, 1, 1, 1, 1]
  lhs = 14
  rhs = 14
  match? True

beta = 2, C = 1, D = 2, mu = [3, 2, 1, 1]
  lhs = 2
  rhs = 2
  match? True

beta = 2, C = 2, D = 0, mu = [3, 2, 2]
  lhs = 2
  rhs = 2
  match? True

Case 2:
beta = 2, C = 1, D = 5, mu = [2, 1, 1, 1, 1, 1]
  lhs = 312
  rhs = 312
  match? True

beta = 2, C = 2, D = 3, mu = [2, 2, 1, 1, 1]
  lhs = 32
  rhs = 32
  match? True

beta = 2, C = 3, D = 1, mu = [2, 2, 2, 1]
  lhs = 8
  